# RB Pipeline v4 Launcher v0.6

Use this notebook to run complete tri-stream preprocessing: `detect -> silhouette -> pack_tri_stream`. The output corpus is written to `training-data-v4-tri-stream` and contains `x_distance_image`, `x_orientation_image`, and `x_geometry`.

In [1]:
from pathlib import Path

import importlib
import sys
from IPython.display import display


def find_pipeline_root(start: Path | None = None) -> Path:
    candidate = (start or Path.cwd()).resolve()
    if candidate.is_file():
        candidate = candidate.parent

    for current in (candidate, *candidate.parents):
        if (current / 'rb_pipeline_v4').exists() and (current / 'input-images').exists():
            return current

        nested = current / '02_synthetic-data-processing-v4.0'
        if (nested / 'rb_pipeline_v4').exists() and (nested / 'input-images').exists():
            return nested

    raise RuntimeError('Could not locate 02_synthetic-data-processing-v4.0 root.')


PIPELINE_ROOT = find_pipeline_root()
if str(PIPELINE_ROOT) not in sys.path:
    sys.path.insert(0, str(PIPELINE_ROOT))

import rb_pipeline_v4.brightness_normalization as brightness_mod
import rb_pipeline_v4.config as config_mod
import rb_pipeline_v4.pack_tri_stream_stage as pack_tri_stream_mod
import rb_pipeline_v4.pipeline as pipeline_mod
import rb_pipeline_v4.widgets as widgets_base_mod
import rb_pipeline_v4.widgets_v05 as widgets_v05_mod
import rb_pipeline_v4.widgets_v06 as widgets_mod

# Keep this launcher honest during iterative UI work in a live kernel.
for module in (
    config_mod,
    brightness_mod,
    pack_tri_stream_mod,
    pipeline_mod,
    widgets_base_mod,
    widgets_v05_mod,
    widgets_mod,
):
    importlib.reload(module)

actual_build = getattr(widgets_mod, 'WIDGETS_UI_BUILD_V06', 'unknown')
print('Loaded widgets module:', widgets_mod.__file__)
print('UI build:', actual_build)

_previous_launcher_closed = False
_previous_launcher = globals().get('_pipeline_launcher_v06_instance')
if _previous_launcher is not None:
    try:
        _previous_launcher.close()
        _previous_launcher_closed = True
    except Exception:
        pass

_previous_widget = globals().get('_pipeline_launcher_v06_widget')
if _previous_widget is not None and not _previous_launcher_closed:
    try:
        _previous_widget.close()
    except Exception:
        pass

launcher = widgets_mod.PipelineLauncherV6(PIPELINE_ROOT)
launcher_widget = launcher.widget
display(launcher_widget)
globals()['_pipeline_launcher_v06_instance'] = launcher
globals()['_pipeline_launcher_v06_widget'] = launcher_widget

print('Preview controls:', type(launcher.preview_sample_offset_input).__name__, '+', type(launcher.preview_refresh_button).__name__)
print('Tri-stream orientation preview:', type(launcher.preview_orientation_image).__name__)


Loaded widgets module: /home/mitch/development/raccoon-ball/02_synthetic-data-processing-v4.0/rb_pipeline_v4/widgets_v06.py
UI build: 2026-04-29-tri-stream-v06-distance-orientation-geometry


Preview controls: BoundedIntText + Button
Tri-stream orientation preview: Image
